In [1]:
# -- same boilerplate / setup as before --------------------------------------
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import os, pickle, math
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from utils.cache_directory_utils import CacheDirectoryManager
from data.load_data import mount_drive, load_data
from models.transformer_model import CaseTransformerModel
from models.transformer_creation import create_vocabularies_from_data
from config import get_model_config
from analysis.pca_vectors import (
    extract_all_pre_classifier_vectors,
    plot_multilabel_cluster_heatmap,
)
from notebook_helpers import prepare_sequences_and_labels

BASE_CACHE  = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR  = BASE_CACHE / 'saved_models'
OUTPUT_DIR  = BASE_CACHE / 'pca_best_fold'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_FOLD_IDX = 0
N_FOLDS       = 5
RANDOM_SEED   = 42
N_CLUSTERS    = 6
bert_out_dim  = 768
cuda_index    = 0
computation_device = torch.device(
    f'cuda:{cuda_index}' if torch.cuda.is_available() else 'cpu'
)

# -- the three labels to show -------------------------------------------------
SELECTED_LABELS = [
    'CHARGES_REFUSED',
    'PLED_NOT_GUILTY_TO_CHARGES',
    'TRIAL_VERDICT_REACHED',
]

model_cache_manager = CacheDirectoryManager(base_path=BASE_CACHE)
model_config = get_model_config()

with open(MODELS_DIR / 'label_processor.pkl', 'rb') as f:
    label_processor = pickle.load(f)
print(f'Loaded label processor ({len(label_processor.label_names)} labels)')

sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
with open(sequences_path, 'rb') as f:
    sequences, label_sequences_raw = pickle.load(f)
print(f'Loaded {len(sequences)} sequences')

filtered_training_data = list(zip(sequences, label_sequences_raw))
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_val_sets = [
    [filtered_training_data[i] for i in val_idx]
    for _, val_idx in kf.split(filtered_training_data)
]

parameters_dict = {
    'model_validation_strategy': 'k_fold_cross_validation',
    'k_fold_num_folds': 5,
    'k_fold_shuffle': True,
    'random_seed': 42,
    'skip_duplicate_labels': False,
    'skip_duplicate_labels_on_import': False,
    'bert_model_to_use': 'bert-base-uncased',
    'bert_batch_size': 32,
    'sampling_strategy': 'balanced',
    'weighted_loss_function': False,
    'use_mixed_precision': True,
    'use_learning_rate_scheduler': True,
    'gradient_clip_norm': 1.0,
    'number_of_epochs_for_training': 20,
    'batch_size_for_training': 16,
    'low_frequency_threshold': 400,
    'folder_name_directory': 'balanced_8',
}

drive_root_folder = mount_drive('tulane-courts-shared/v2')
court_data = load_data(drive_root_folder)
charge_vocab, mag_vocab = create_vocabularies_from_data(court_data)
del court_data

bond_dim = model_config.NUM_BOND_CLASSES
sample_pt = MODELS_DIR / f'k_fold_fold_{BEST_FOLD_IDX + 1}' / \
            f'final_model_mlb_fold_{BEST_FOLD_IDX + 1}.pt'
sample_sd = torch.load(sample_pt, map_location='cpu')
fuse_in   = sample_sd['fuse.0.weight'].shape[1]
FIXED_DIMS = 32 + 192 + 768 + 32 + 96 + 192 + 1 + 1
correct_temporal_dim = fuse_in - FIXED_DIMS - (bond_dim * 2)

model = CaseTransformerModel(
    charge_vocab=charge_vocab,
    mag_vocab=mag_vocab,
    bert_cache_dir=model_cache_manager.get_cache_dir('bert'),
    bert_out_dim=bert_out_dim,
    num_labels=model_config.num_labels,
    bond_dim=bond_dim,
    temporal_dim=correct_temporal_dim,
    cache_manager=model_cache_manager,
)
model.load_state_dict(torch.load(sample_pt, map_location=computation_device))
model.eval()
model.to(computation_device)

_saved_min_year = model_config.min_year
_saved_max_year = model_config.max_year
model_config.update_date_range(2008, 2025)

try:
    vectors, final_labels, case_labels_matrix, metadata = \
        extract_all_pre_classifier_vectors(
            model,
            fold_val_sets[BEST_FOLD_IDX],
            label_processor,
            computation_device,
            use_teacher_forcing=True,
            verbose=True,
        )
finally:
    model_config.update_date_range(_saved_min_year, _saved_max_year)

del model
torch.cuda.empty_cache()

vectors = StandardScaler().fit_transform(vectors)
vectors = normalize(vectors, norm='l2')

case_labels_arr = np.asarray(case_labels_matrix)

pca = PCA(n_components=50, random_state=RANDOM_SEED)
vecs_50d = pca.fit_transform(vectors)
explained = pca.explained_variance_ratio_.sum() * 100

projected_2d = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate='auto',
    init='pca',
    random_state=RANDOM_SEED,
).fit_transform(vecs_50d)

km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km_labels = km.fit_predict(projected_2d)

# -- filter label matrix and names to only the 3 selected labels --------------
selected_indices = [
    label_processor.label_names.index(lbl)
    for lbl in SELECTED_LABELS
]
selected_label_matrix = case_labels_arr[:, selected_indices]

# -- set ONCE at the top of your script, before any plotting ------------------
plt.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          8,
    'axes.titlesize':     8,
    'axes.labelsize':     8,
    'xtick.labelsize':    7,
    'ytick.labelsize':    7,
    'legend.fontsize':    7,
    'figure.titlesize':   9,
    'axes.linewidth':     0.5,
    'xtick.major.width':  0.5,
    'ytick.major.width':  0.5,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
    'pdf.fonttype':       42,   # editable text if you ever export PDF
    'ps.fonttype':        42,
})


def plot_label_grid_colored_by_cluster(
    projected_2d, label_matrix, label_names, cluster_labels, output_path,
    title='t-SNE Embeddings (Best Fold) - Label Presence', max_cols=1,
):
    label_matrix   = np.asarray(label_matrix)
    cluster_labels = np.asarray(cluster_labels)
    n_samples, n_labels = label_matrix.shape
    unique_clusters = sorted(np.unique(cluster_labels))
    n_real = sum(1 for c in unique_clusters if c != -1)
    cmap = cm.get_cmap('tab10', max(n_real, 1))
    cl_to_idx = {cl: i for i, cl in enumerate(unique_clusters)}

    n_cols = min(max_cols, n_labels)
    n_rows = math.ceil(n_labels / n_cols)

    # -- IEEE single-column: 3.5" wide; ~2.8" per panel tall -----------------
    col_width_in = 3.5
    panel_h_in   = 2.8
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(col_width_in, panel_h_in * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).flatten()

    for i, name in enumerate(label_names):
        ax = axes[i]
        mask_pos = label_matrix[:, i] == 1
        n_pos    = int(mask_pos.sum())
        pct      = 100 * n_pos / n_samples if n_samples else 0.0

        # background (negative) - small, grey
        ax.scatter(
            projected_2d[~mask_pos, 0], projected_2d[~mask_pos, 1],
            c='0.45', s=5, alpha=0.6, edgecolors='none', rasterized=True,
        )

        # positive points coloured by cluster
        for cl in unique_clusters:
            mask = mask_pos & (cluster_labels == cl)
            if not mask.any():
                continue
            fc = 'dimgrey' if cl == -1 else [cmap(cl_to_idx[cl])]
            ax.scatter(
                projected_2d[mask, 0], projected_2d[mask, 1],
                c=fc, s=10, alpha=0.9,
                edgecolors='white', linewidths=0.2, rasterized=True,
            )

        # clean label: replace underscores with spaces for readability
        clean_name = name.replace('_', ' ')
        ax.set_title(f'{clean_name}\n(n={n_pos:,},  {pct:.1f}%)',
                     fontsize=8, fontweight='bold', pad=4)
        ax.set_xlabel('t-SNE dim 1', fontsize=7)
        ax.set_ylabel('t-SNE dim 2', fontsize=7)
        ax.tick_params(labelsize=6, length=2, width=0.5)
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    for j in range(n_labels, len(axes)):
        axes[j].set_visible(False)

    # -- compact horizontal legend below all panels ----------------------------
    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='dimgrey' if cl == -1 else cmap(cl_to_idx[cl]),
                   markeredgecolor='none', markersize=5,
                   label='Noise' if cl == -1 else f'Cluster {cl}')
        for cl in unique_clusters
    ]
    fig.legend(
        handles=handles,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.04),
        ncol=len(unique_clusters),
        fontsize=7,
        frameon=True,
        edgecolor='grey',
        fancybox=False,
        handletextpad=0.3,
        columnspacing=0.8,
    )

    fig.suptitle(title, fontsize=9, fontweight='bold')

    fig.savefig(output_path, dpi=600, bbox_inches='tight', format='png')
    print(f'Saved: {output_path}')
    plt.close(fig)


# -- call - note max_cols=1 forces vertical stacking --------------------------
fold_label = f'Fold {BEST_FOLD_IDX + 1} (best)'

plot_label_grid_colored_by_cluster(
    projected_2d=projected_2d,
    label_matrix=selected_label_matrix,
    label_names=SELECTED_LABELS,
    cluster_labels=km_labels,
    output_path=OUTPUT_DIR / 'tsne_label_grid_selected_ones.png',
    title=f't-SNE Embeddings ({fold_label})\nLabel Presence by Cluster',
    max_cols=1,     # <- stacks vertically to fit a 3.5" column
)

Loaded label processor (20 labels)
Loaded 2533 sequences
[OK] Found Google Drive mount at /data/krosen5/gdrive-all
[OK] Google Drive mount is accessible
[OK] Data directory found: /data/krosen5/gdrive-all/tulane-courts-shared/v2
Creating charge vocabulary from charges DataFrame with 2139 unique charges
Found 5 unique magistrate sections in mag data.
Creating magistrate vocabulary of size 5
Date range updated: 2008 to 2025


Extracting pre-classifier vectors for 507 samples: 100%|##########| 507/507 [00:04<00:00, 108.63it/s]


Extracted 507 vectors of dimension 896
Date range updated: 1990 to 2025
Saved: /data/krosen5/cache/transformers/balanced_8/pca_best_fold/tsne_label_grid_selected_ones.png
